# AqSolDB LogS Prediction Pipeline

**Project type:** Computational chemistry / QSAR / machine learning  
**Goal:** Predict aqueous solubility (LogS) from molecular structure using curated descriptors, Morgan fingerprints, and model benchmarking.


## What this notebook does

1. Loads and cleans the AqSolDB dataset.
2. Computes physicochemical descriptors and Morgan fingerprints.
3. Benchmarks fingerprint generation speed.
4. Runs exploratory data analysis.
5. Splits the data into train, validation, and test sets.
6. Trains and compares five regression models across three feature sets.
7. Visualizes performance, saves the best model, and runs inference on example molecules.


In [ ]:
# Imports & global configuration

import warnings
import time
import os
import pickle
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, SaltRemover
)
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = {
    "primary":   "#2C6E8A",
    "secondary": "#E07B3F",
    "accent":    "#5BAD72",
    "highlight": "#B5446E",
    "neutral":   "#7A7A8C",
    "bg":        "#F8F9FA",
}
MODEL_COLORS = [
    "#2C6E8A", "#E07B3F", "#5BAD72", "#B5446E", "#7A7A8C"
]

OUTPUT_DIR = Path("pipeline_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
FP_RADIUS    = 2
FP_NBITS     = 2048

print("✓ All imports successful. Output directory:", OUTPUT_DIR.resolve())


## 1. Data loading and molecular cleaning

This section reads the raw AqSolDB file, renames the target column to `LogS`, and standardizes the SMILES strings with RDKit. Invalid structures are removed so they do not contaminate downstream feature engineering and model training.


In [ ]:
def load_dataset(path: str) -> pd.DataFrame:
    """Load the AqSolDB CSV dataset."""
    df = pd.read_csv(path)
    df = df.rename(columns={"Solubility": "LogS"})
    print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
    print(f"LogS range : [{df['LogS'].min():.3f}, {df['LogS'].max():.3f}]")
    print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    return df


class MolecularCleaner:
    """RDKit-based molecular standardisation pipeline."""

    def __init__(self):
        self._remover   = SaltRemover.SaltRemover()
        self._uncharger = rdMolStandardize.Uncharger()

    def clean_smiles(self, smiles: str):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        try:
            mol = self._remover.StripMol(mol, dontRemoveEverything=True)
            mol = self._uncharger.uncharge(mol)
        except Exception:
            return None
        return Chem.MolToSmiles(mol, canonical=True)

    def clean_dataframe(self, df: pd.DataFrame, smiles_col: str = "SMILES") -> pd.DataFrame:
        n_before = len(df)
        df = df.copy()
        df["clean_smiles"] = df[smiles_col].apply(self.clean_smiles)

        n_invalid = df["clean_smiles"].isna().sum()
        df = df.dropna(subset=["clean_smiles"]).reset_index(drop=True)
        n_after = len(df)

        print(f"Molecules before cleaning : {n_before:,}")
        print(f"Invalid / removed         : {n_invalid:,}")
        print(f"Molecules after cleaning  : {n_after:,}")
        return df


## 2. Feature extraction and fingerprint benchmarking

This section converts each molecule into two complementary representations: interpretable physicochemical descriptors and hashed Morgan fingerprints. It also compares a slow loop-based fingerprint routine with a more efficient vectorised approach.


In [ ]:
DESCRIPTOR_FUNCTIONS = {
    "MolLogP":        Descriptors.MolLogP,
    "MolWt":          Descriptors.MolWt,
    "TPSA":           Descriptors.TPSA,
    "NumHDonors":     rdMolDescriptors.CalcNumHBD,
    "NumHAcceptors":  rdMolDescriptors.CalcNumHBA,
    "NumRotBonds":    rdMolDescriptors.CalcNumRotatableBonds,
    "NumArRings":     rdMolDescriptors.CalcNumAromaticRings,
    "NumSatRings":    rdMolDescriptors.CalcNumSaturatedRings,
    "RingCount":      rdMolDescriptors.CalcNumRings,
    "HeavyAtomCount": Descriptors.HeavyAtomCount,
    "MolMR":          Descriptors.MolMR,
    "LabuteASA":      rdMolDescriptors.CalcLabuteASA,
    "FractionCSP3":   rdMolDescriptors.CalcFractionCSP3,
    "NumHeteroatoms": rdMolDescriptors.CalcNumHeteroatoms,
    "BertzCT":        Descriptors.BertzCT,
}

PHYS_COLS = list(DESCRIPTOR_FUNCTIONS.keys())


def compute_physical_descriptors(smiles_list: list) -> pd.DataFrame:
    records = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            records.append({k: np.nan for k in DESCRIPTOR_FUNCTIONS})
            continue
        row = {}
        for name, fn in DESCRIPTOR_FUNCTIONS.items():
            try:
                row[name] = fn(mol)
            except Exception:
                row[name] = np.nan
        records.append(row)
    return pd.DataFrame(records, columns=list(DESCRIPTOR_FUNCTIONS.keys()))


def fingerprints_loop(smiles_list: list, radius: int = FP_RADIUS, n_bits: int = FP_NBITS) -> np.ndarray:
    gen = GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = np.zeros((len(smiles_list), n_bits), dtype=np.uint8)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            bv = gen.GetFingerprint(mol)
            DataStructs.ConvertToNumpyArray(bv, fps[i])
    return fps


def fingerprints_vectorised(smiles_list: list, radius: int = FP_RADIUS, n_bits: int = FP_NBITS) -> np.ndarray:
    gen    = GetMorganGenerator(radius=radius, fpSize=n_bits)
    mols   = [Chem.MolFromSmiles(s) for s in smiles_list]
    fps    = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, mol in enumerate(mols):
        if mol is not None:
            DataStructs.ConvertToNumpyArray(gen.GetFingerprint(mol), fps[i])
    return fps


def benchmark_fingerprint_methods(smiles_list: list, n_repeats: int = 3) -> dict:
    loop_times, vec_times = [], []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        fp_loop = fingerprints_loop(smiles_list)
        loop_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        fp_vec  = fingerprints_vectorised(smiles_list)
        vec_times.append(time.perf_counter() - t0)

    t_loop  = min(loop_times)
    t_vec   = min(vec_times)
    speedup = t_loop / t_vec
    print(f"Loop method   : {t_loop:.3f} s")
    print(f"Vectorised    : {t_vec:.3f} s")
    print(f"Speed-up      : {speedup:.2f}×")
    assert np.array_equal(fp_loop, fp_vec), "Fingerprints mismatch!"
    return {"loop_time": t_loop, "vec_time": t_vec, "speedup": speedup}


## 3. Exploratory data analysis

The plots in this section show how the target variable is distributed, how cleaning changes the dataset, how descriptors relate to LogS, and which physicochemical features are most informative.


In [ ]:
def plot_cleaning_impact(df_raw: pd.DataFrame, df_clean: pd.DataFrame, save_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=PALETTE["bg"])
    fig.suptitle("Dataset State: Before vs. After Molecular Cleaning",
                 fontsize=14, fontweight="bold", y=1.01)

    ax = axes[0]
    ax.set_facecolor(PALETTE["bg"])
    sns.kdeplot(df_raw["LogS"],  ax=ax, color=PALETTE["highlight"],
                label=f"Raw (n={len(df_raw):,})", linewidth=2, fill=True,
                alpha=0.25)
    sns.kdeplot(df_clean["LogS"], ax=ax, color=PALETTE["primary"],
                label=f"Cleaned (n={len(df_clean):,})", linewidth=2,
                fill=True, alpha=0.25)
    ax.set_xlabel("LogS (log mol/L)", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title("LogS Distribution Shift", fontsize=12)
    ax.legend(fontsize=10)
    ax.axvline(df_clean["LogS"].mean(), color=PALETTE["primary"],
               linestyle="--", alpha=0.7, label="Clean mean")

    ax2 = axes[1]
    ax2.set_facecolor(PALETTE["bg"])
    labels = ["Raw", "Cleaned"]
    counts = [len(df_raw), len(df_clean)]
    bars   = ax2.bar(labels, counts,
                     color=[PALETTE["highlight"], PALETTE["primary"]],
                     width=0.45, edgecolor="white", linewidth=1.5)
    for bar, cnt in zip(bars, counts):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 30, f"{cnt:,}",
                 ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax2.set_ylabel("Molecule Count", fontsize=11)
    ax2.set_title("Molecules Retained After Cleaning", fontsize=12)
    ax2.set_ylim(0, max(counts) * 1.15)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_logsS_distribution(logsS: pd.Series, save_path: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=PALETTE["bg"])
    fig.suptitle("Target Variable Analysis: Aqueous Solubility (LogS)",
                 fontsize=14, fontweight="bold")

    ax = axes[0]
    ax.set_facecolor(PALETTE["bg"])
    sns.histplot(logsS, ax=ax, bins=60, color=PALETTE["primary"],
                 alpha=0.75, edgecolor="white", stat="density")
    sns.kdeplot(logsS, ax=ax, color=PALETTE["secondary"],
                linewidth=2.5, label="KDE")
    ax.axvline(logsS.mean(),   color=PALETTE["highlight"],
               linestyle="--", linewidth=1.8, label=f"Mean={logsS.mean():.2f}")
    ax.axvline(logsS.median(), color=PALETTE["accent"],
               linestyle=":",  linewidth=1.8, label=f"Median={logsS.median():.2f}")
    ax.set_xlabel("LogS (log mol/L)", fontsize=11)
    ax.set_ylabel("Density",          fontsize=11)
    ax.set_title("Distribution of LogS", fontsize=12)
    ax.legend(fontsize=10)

    stats_text = (f"n     = {len(logsS):,}\n"
                  f"Mean  = {logsS.mean():.3f}\n"
                  f"Std   = {logsS.std():.3f}\n"
                  f"Min   = {logsS.min():.3f}\n"
                  f"Max   = {logsS.max():.3f}\n"
                  f"Skew  = {logsS.skew():.3f}")
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", alpha=0.8))

    ax2 = axes[1]
    ax2.set_facecolor(PALETTE["bg"])
    bins_   = [-np.inf, -6, -4, -2, np.inf]
    labels_ = ["Insoluble\n(<-6)", "Poorly\n(-6 to -4)",
                "Moderately\n(-4 to -2)", "Soluble\n(>-2)"]
    cats    = pd.cut(logsS, bins=bins_, labels=labels_)
    counts  = cats.value_counts().sort_index()
    bars    = ax2.bar(labels_, counts.values,
                      color=[PALETTE["highlight"], PALETTE["secondary"],
                             PALETTE["primary"],   PALETTE["accent"]],
                      edgecolor="white", linewidth=1.5)
    for bar, cnt in zip(bars, counts.values):
        pct = cnt / len(logsS) * 100
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 20,
                 f"{cnt:,}\n({pct:.1f}%)",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax2.set_ylabel("Count", fontsize=11)
    ax2.set_title("Solubility Class Distribution", fontsize=12)
    ax2.set_ylim(0, counts.max() * 1.25)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_correlation_heatmap(phys_df: pd.DataFrame, logsS: pd.Series, save_path: Path) -> None:
    combined = phys_df.copy()
    combined["LogS"] = logsS.values
    corr = combined.corr()

    fig, ax = plt.subplots(figsize=(14, 11), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    mask = np.triu(np.ones_like(corr, dtype=bool))
    cmap = sns.diverging_palette(230, 20, as_cmap=True)
    sns.heatmap(corr, mask=mask, ax=ax, cmap=cmap,
                vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 7},
                square=True, linewidths=0.4,
                cbar_kws={"shrink": 0.7, "label": "Pearson r"})

    ax.set_title("Pearson Correlation — Physical Descriptors + LogS",
                 fontsize=13, fontweight="bold", pad=16)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_fingerprint_benchmark(bench: dict, save_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    methods = ["Loop\n(sequential)", "Vectorised\n(pre-allocated)"]
    times   = [bench["loop_time"], bench["vec_time"]]
    colors  = [PALETTE["highlight"], PALETTE["primary"]]

    bars = ax.bar(methods, times, color=colors,
                  width=0.42, edgecolor="white", linewidth=1.5)
    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01, f"{t:.3f} s",
                ha="center", va="bottom", fontsize=11, fontweight="bold")

    ax.set_ylabel("Wall-clock time (seconds)", fontsize=11)
    ax.set_title(
        f"Fingerprint Generation: Speed Benchmark\n"
        f"Speed-up: {bench['speedup']:.2f}× (n={FP_NBITS} bits, radius={FP_RADIUS})",
        fontsize=12, fontweight="bold"
    )
    ax.set_ylim(0, max(times) * 1.35)

    ax.annotate(f"↓ {bench['speedup']:.2f}×\nfaster",
                xy=(1, bench["vec_time"]),
                xytext=(0.65, (bench["loop_time"] + bench["vec_time"]) / 2),
                arrowprops=dict(arrowstyle="->", color=PALETTE["accent"], lw=2),
                fontsize=11, color=PALETTE["accent"], fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_descriptor_vs_logsS(phys_df: pd.DataFrame, logsS: pd.Series, save_path: Path, top_n: int = 6) -> None:
    corrs  = phys_df.corrwith(logsS).abs().sort_values(ascending=False)
    top    = corrs.head(top_n).index.tolist()

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor=PALETTE["bg"])
    fig.suptitle(f"Top {top_n} Descriptors vs LogS", fontsize=14, fontweight="bold")

    for ax, col in zip(axes.flat, top):
        ax.set_facecolor(PALETTE["bg"])
        ax.scatter(phys_df[col], logsS, alpha=0.25, s=6,
                   color=PALETTE["primary"], rasterized=True)
        r = np.corrcoef(phys_df[col].fillna(0), logsS)[0, 1]
        ax.set_xlabel(col, fontsize=10)
        ax.set_ylabel("LogS", fontsize=10)
        ax.set_title(f"r = {r:.3f}", fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


## 4. Train / validation / test split

A three-way split is used so the validation set can guide model selection while the test set remains untouched until the end. That avoids accidental leakage and gives a more honest performance estimate.


In [ ]:
def three_way_split(X: np.ndarray, y: np.ndarray, train_frac: float = 0.70, val_frac: float = 0.15, random_state: int = RANDOM_STATE) -> tuple:
    test_frac = 1.0 - train_frac - val_frac
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=test_frac, random_state=random_state
    )
    val_relative = val_frac / (train_frac + val_frac)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=val_relative, random_state=random_state
    )
    print(f"  Train : {X_train.shape[0]:,}  ({X_train.shape[0]/len(y)*100:.1f}%)")
    print(f"  Val   : {X_val.shape[0]:,}   ({X_val.shape[0]/len(y)*100:.1f}%)")
    print(f"  Test  : {X_test.shape[0]:,}   ({X_test.shape[0]/len(y)*100:.1f}%)")
    return X_train, X_val, X_test, y_train, y_val, y_test


## 5. Model benchmarking

Five models are trained on three feature sets: physical descriptors, Morgan fingerprints, and a hybrid combination of both. The outputs are compared using R², RMSE, and MAE.


In [ ]:
def build_models(random_state: int = RANDOM_STATE) -> dict:
    models = {
        "Ridge Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model",  Ridge(alpha=1.0))
        ]),
        "SVR": Pipeline([
            ("scaler", StandardScaler()),
            ("model",  SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale"))
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("model",  RandomForestRegressor(
                n_estimators=300, max_features="sqrt",
                min_samples_leaf=2, n_jobs=-1,
                random_state=random_state
            ))
        ]),
        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("model",  XGBRegressor(
                n_estimators=500, learning_rate=0.05,
                max_depth=6, subsample=0.8,
                colsample_bytree=0.8, n_jobs=-1,
                random_state=random_state, verbosity=0
            ))
        ]),
        "LightGBM": Pipeline([
            ("scaler", StandardScaler()),
            ("model",  LGBMRegressor(
                n_estimators=500, learning_rate=0.05,
                num_leaves=63, subsample=0.8,
                colsample_bytree=0.8, n_jobs=-1,
                random_state=random_state, verbose=-1
            ))
        ]),
    }
    return models


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "R2":   r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE":  mean_absolute_error(y_true, y_pred),
    }


def train_and_evaluate(models: dict, X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, X_test: np.ndarray, y_test: np.ndarray, feature_set_name: str) -> pd.DataFrame:
    records = []
    print(f"\n{'─'*56}")
    print(f"  Feature Set: {feature_set_name}")
    print(f"{'─'*56}")
    for name, pipe in models.items():
        t0 = time.perf_counter()
        pipe.fit(X_train, y_train)
        t_fit = time.perf_counter() - t0

        val_metrics  = evaluate(y_val,  pipe.predict(X_val))
        test_metrics = evaluate(y_test, pipe.predict(X_test))

        print(f"  {name:<22} | Val  R²={val_metrics['R2']:.3f} RMSE={val_metrics['RMSE']:.3f} | Test R²={test_metrics['R2']:.3f} RMSE={test_metrics['RMSE']:.3f} | {t_fit:.1f}s")

        records.append({
            "Model":       name,
            "Feature Set": feature_set_name,
            **{f"Val_{k}": v  for k, v in val_metrics.items()},
            **{f"Test_{k}": v for k, v in test_metrics.items()},
        })
    return pd.DataFrame(records)


## 6. Results, plots, and interpretation

This section turns the benchmark output into readable tables and charts. It also generates predicted-vs-actual plots and feature-importance summaries for the strongest model.


In [ ]:
def plot_benchmark_summary(results_df: pd.DataFrame, save_path:  Path) -> None:
    pivot = results_df.pivot(index="Model", columns="Feature Set", values="Test_R2")

    fig, ax = plt.subplots(figsize=(13, 6), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    n_models = len(pivot)
    n_sets   = len(pivot.columns)
    bar_w    = 0.22
    x        = np.arange(n_models)
    offsets  = np.linspace(-(n_sets - 1) / 2 * bar_w, (n_sets - 1) / 2 * bar_w, n_sets)

    fs_colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]

    for i, (fs, col) in enumerate(zip(pivot.columns, fs_colors)):
        vals = pivot[fs].values
        bars = ax.bar(x + offsets[i], vals, width=bar_w, label=fs, color=col, edgecolor="white", linewidth=1)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.2f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, fontsize=10)
    ax.set_ylabel("Test R²", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.9, color="grey", linewidth=0.8, linestyle="--", alpha=0.6, label="R²=0.9 reference")
    ax.set_title("Model Benchmarking: Test R² Across Feature Sets", fontsize=13, fontweight="bold")
    ax.legend(title="Feature Set", fontsize=9, title_fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_rmse_comparison(results_df: pd.DataFrame, save_path:  Path) -> None:
    pivot = results_df.pivot(index="Model", columns="Feature Set", values="Test_RMSE")

    fig, ax = plt.subplots(figsize=(10, 5), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    markers = ["o", "s", "^"] 
    fs_colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]

    for i, (fs, col, mrk) in enumerate(zip(pivot.columns, fs_colors, markers)):
        ax.plot(pivot.index, pivot[fs], marker=mrk, label=fs, color=col, linewidth=2, markersize=9)

    ax.set_ylabel("Test RMSE (log mol/L)", fontsize=11)
    ax.set_title("Test RMSE Comparison Across Models & Feature Sets", fontsize=12, fontweight="bold")
    ax.legend(title="Feature Set", fontsize=9)
    ax.tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_predicted_vs_actual(y_true: np.ndarray, y_pred: np.ndarray, model_name: str, feature_set_name: str, save_path: Path) -> None:
    metrics = evaluate(y_true, y_pred)
    residuals = y_true - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor=PALETTE["bg"])
    fig.suptitle(
        f"Best Model: {model_name} | {feature_set_name}  (R²={metrics['R2']:.3f}, RMSE={metrics['RMSE']:.3f})",
        fontsize=13, fontweight="bold"
    )

    ax = axes[0]
    ax.set_facecolor(PALETTE["bg"])
    lo, hi = min(y_true.min(), y_pred.min()) - 0.5, max(y_true.max(), y_pred.max()) + 0.5
    ax.scatter(y_true, y_pred, alpha=0.3, s=8, color=PALETTE["primary"], rasterized=True)
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.5, label="Identity")
    ax.fill_between([lo, hi], [lo - 1, hi - 1], [lo + 1, hi + 1], alpha=0.08, color=PALETTE["secondary"], label="±1 log unit band")
    ax.set_xlabel("Actual LogS", fontsize=11)
    ax.set_ylabel("Predicted LogS", fontsize=11)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.legend(fontsize=9)
    ax.set_title("Predicted vs Actual", fontsize=12)

    stats_txt = (f"R²   = {metrics['R2']:.4f}\nRMSE = {metrics['RMSE']:.4f}\nMAE  = {metrics['MAE']:.4f}")
    ax.text(0.04, 0.96, stats_txt, transform=ax.transAxes, va="top", fontsize=9, bbox=dict(boxstyle="round,pad=0.4", fc="white", alpha=0.85))

    ax2 = axes[1]
    ax2.set_facecolor(PALETTE["bg"])
    sns.histplot(residuals, ax=ax2, bins=60, color=PALETTE["primary"], alpha=0.7, edgecolor="white", stat="density")
    sns.kdeplot(residuals, ax=ax2, color=PALETTE["secondary"], linewidth=2.5)
    ax2.axvline(0, color="black", linewidth=1.5, linestyle="--")
    ax2.set_xlabel("Residual (Actual − Predicted)", fontsize=11)
    ax2.set_ylabel("Density", fontsize=11)
    ax2.set_title("Residual Distribution", fontsize=12)
    ax2.text(0.97, 0.97, f"Mean  = {residuals.mean():.3f}\nStd   = {residuals.std():.3f}", transform=ax2.transAxes, ha="right", va="top", fontsize=9, bbox=dict(boxstyle="round,pad=0.4", fc="white", alpha=0.85))

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def plot_feature_importance(model_pipe: object, feature_names: list, X_val: np.ndarray, y_val: np.ndarray, model_name: str, save_path:  Path, top_n: int = 25) -> None:
    model = model_pipe.named_steps["model"]

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        imp_type    = "Built-in Importance (Gain/MDI)"
    elif hasattr(model, "coef_"):
        importances = np.abs(model.coef_)
        imp_type    = "|Coefficient| (Ridge)"
    else:
        r = permutation_importance(model_pipe, X_val, y_val, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
        importances = r.importances_mean
        imp_type    = "Permutation Importance (Val)"

    idx    = np.argsort(importances)[::-1][:top_n]
    top_names  = [feature_names[i] for i in idx]
    top_vals   = importances[idx].astype(np.float64)
    top_vals  /= top_vals.max()

    fig, ax = plt.subplots(figsize=(9, max(5, top_n * 0.32)), facecolor=PALETTE["bg"])
    ax.set_facecolor(PALETTE["bg"])

    colors_ = [PALETTE["primary"] if v > 0.5 else PALETTE["secondary"] for v in top_vals]
    bars = ax.barh(range(top_n)[::-1], top_vals, color=colors_, edgecolor="white", linewidth=0.8)
    ax.set_yticks(range(top_n)[::-1])
    ax.set_yticklabels(top_names, fontsize=8)
    ax.set_xlabel(f"Normalised {imp_type}", fontsize=10)
    ax.set_title(f"Feature Importance — {model_name} (Top {top_n})", fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"  ✓ Saved → {save_path}")


def print_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    summary = results_df[["Model", "Feature Set", "Test_R2", "Test_RMSE", "Test_MAE"]].copy()
    summary.columns = ["Model", "Feature Set", "R²", "RMSE", "MAE"]
    summary = summary.sort_values(["Feature Set", "R²"], ascending=[True, False])
    summary[["R²", "RMSE", "MAE"]] = summary[["R²", "RMSE", "MAE"]].round(4)

    print("\n" + "═" * 70)
    print(" MODEL BENCHMARKING SUMMARY — TEST SET")
    print("═" * 70)
    print(summary.to_string(index=False))
    print("═" * 70)
    return summary


## 7. Model saving and inference

After benchmarking, the best pipeline is saved to disk and reused to predict LogS for new molecules. The inference function accepts a SMILES string and returns a solubility estimate.


In [ ]:
def save_model(pipe: object, scaler_phys: StandardScaler, path: str = "best_logsS_model.joblib") -> None:
    payload = {"pipeline": pipe, "scaler_phys": scaler_phys}
    joblib.dump(payload, path, compress=3)
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✓ Model saved → {path}  ({size_kb:.1f} KB)")


def predict_logsS(smiles: str, model_path: str = "best_logsS_model.joblib") -> float:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles!r}")

    phys_row = {}
    for name, fn in DESCRIPTOR_FUNCTIONS.items():
        try:
            phys_row[name] = fn(mol)
        except Exception:
            phys_row[name] = 0.0
    phys_arr = np.array([list(phys_row.values())], dtype=np.float64)

    gen    = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)
    fp_arr = np.zeros((1, FP_NBITS), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(gen.GetFingerprint(mol), fp_arr[0])

    X_query = np.hstack([phys_arr, fp_arr]).astype(np.float64)

    payload   = joblib.load(model_path)
    pipe      = payload["pipeline"]
    predicted = pipe.predict(X_query)[0]

    print(f"  SMILES   : {smiles}")
    print(f"  MolWt    : {Descriptors.MolWt(mol):.2f} g/mol")
    print(f"  MolLogP  : {Descriptors.MolLogP(mol):.3f}")
    print(f"  Predicted LogS: {predicted:.4f} log(mol/L)")
    return float(predicted)


## 8. Main execution

Run the notebook from top to bottom, or execute `main()` to reproduce the full pipeline end to end.


In [ ]:
def main(data_path: str = "/mnt/user-data/uploads/curated-solubility-dataset.csv"):
    print("\n" + "═"*60)
    print(" STEP 1 — DATA LOADING & MOLECULAR CLEANING")
    print("═"*60)

    df_raw   = load_dataset(data_path)
    cleaner  = MolecularCleaner()
    df_clean = cleaner.clean_dataframe(df_raw, smiles_col="SMILES")

    plot_cleaning_impact(
        df_raw, df_clean,
        save_path=OUTPUT_DIR / "01_cleaning_impact.png"
    )

    print("\n" + "═"*60)
    print(" STEP 2 — FEATURE EXTRACTION")
    print("═"*60)

    smiles_list = df_clean["clean_smiles"].tolist()
    y           = df_clean["LogS"].values

    print("\n[A] Computing physical descriptors…")
    t0 = time.time()
    phys_df = compute_physical_descriptors(smiles_list)
    phys_df.fillna(phys_df.median(), inplace=True)
    print(f"    Done — {phys_df.shape[1]} descriptors, {time.time()-t0:.1f}s")

    print("\n[B] Fingerprint benchmark (loop vs vectorised)…")
    bench = benchmark_fingerprint_methods(smiles_list[:2000], n_repeats=3)
    plot_fingerprint_benchmark(
        bench,
        save_path=OUTPUT_DIR / "02_fp_benchmark.png"
    )

    print("\n[C] Generating final Morgan fingerprints (vectorised)…")
    t0 = time.time()
    fp_arr = fingerprints_vectorised(smiles_list)
    print(f"    Done — shape={fp_arr.shape}, {time.time()-t0:.1f}s")

    print("\n" + "═"*60)
    print(" STEP 3 — EXPLORATORY DATA ANALYSIS")
    print("═"*60)

    plot_logsS_distribution(
        df_clean["LogS"],
        save_path=OUTPUT_DIR / "03_logsS_distribution.png"
    )
    plot_correlation_heatmap(
        phys_df, df_clean["LogS"].reset_index(drop=True),
        save_path=OUTPUT_DIR / "04_correlation_heatmap.png"
    )
    plot_descriptor_vs_logsS(
        phys_df, pd.Series(y),
        save_path=OUTPUT_DIR / "05_top_descriptors_vs_logsS.png"
    )

    print("\n" + "═"*60)
    print(" STEP 4 — DATA SPLITS")
    print("═"*60)

    X_phys = phys_df.values.astype(np.float64)
    X_fp   = fp_arr.astype(np.float64)
    X_hyb  = np.hstack([X_phys, X_fp])

    phys_feat_names = PHYS_COLS
    fp_feat_names   = [f"FP_{i}" for i in range(FP_NBITS)]
    hyb_feat_names  = phys_feat_names + fp_feat_names

    feature_sets = {
        "Physical":    (X_phys, phys_feat_names),
        "Fingerprint": (X_fp,   fp_feat_names),
        "Hybrid":      (X_hyb,  hyb_feat_names),
    }

    splits = {}
    for fs_name, (X, _) in feature_sets.items():
        print(f"\n  [{fs_name}]")
        splits[fs_name] = three_way_split(X, y)

    print("\n" + "═"*60)
    print(" STEP 5 — MODEL BENCHMARKING")
    print("═"*60)

    all_results = []
    fitted_models = {}

    for fs_name, (X, feat_names) in feature_sets.items():
        X_tr, X_v, X_te, y_tr, y_v, y_te = splits[fs_name]
        models  = build_models()
        results = train_and_evaluate(
            models, X_tr, y_tr, X_v, y_v, X_te, y_te,
            feature_set_name=fs_name
        )
        all_results.append(results)
        for name, pipe in models.items():
            fitted_models[(name, fs_name)] = pipe

    results_df = pd.concat(all_results, ignore_index=True)

    print("\n" + "═"*60)
    print(" STEP 6 — ANALYSIS")
    print("═"*60)

    summary = print_summary_table(results_df)
    summary.to_csv(OUTPUT_DIR / "06_model_summary.csv", index=False)
    print(f"  ✓ Summary saved → {OUTPUT_DIR / '06_model_summary.csv'}")

    plot_benchmark_summary(
        results_df,
        save_path=OUTPUT_DIR / "07_benchmark_r2.png"
    )
    plot_rmse_comparison(
        results_df,
        save_path=OUTPUT_DIR / "08_benchmark_rmse.png"
    )

    best_row   = results_df.loc[results_df["Test_R2"].idxmax()]
    best_name  = best_row["Model"]
    best_fs    = best_row["Feature Set"]
    best_pipe  = fitted_models[(best_name, best_fs)]
    best_feats = feature_sets[best_fs][1]

    X_tr, X_v, X_te, y_tr, y_v, y_te = splits[best_fs]
    y_pred_test = best_pipe.predict(X_te)

    print(f"\n  Best Model  : {best_name}")
    print(f"  Feature Set : {best_fs}")
    print(f"  Test R²     : {best_row['Test_R2']:.4f}")
    print(f"  Test RMSE   : {best_row['Test_RMSE']:.4f}")

    plot_predicted_vs_actual(
        y_te, y_pred_test,
        model_name=best_name,
        feature_set_name=best_fs,
        save_path=OUTPUT_DIR / "09_predicted_vs_actual.png"
    )
    plot_feature_importance(
        best_pipe, best_feats,
        X_v, y_v,
        model_name=f"{best_name} ({best_fs})",
        save_path=OUTPUT_DIR / "10_feature_importance.png"
    )

    print("\n" + "═"*60)
    print(" STEP 7 — SERIALISE MODEL & INFERENCE")
    print("═"*60)

    model_save_path = str(OUTPUT_DIR / "best_logsS_model.joblib")
    scaler_dummy    = StandardScaler()
    save_model(best_pipe, scaler_dummy, path=model_save_path)

    print("\n  Running inference on demo molecules…")
    demo_molecules = {
        "Aspirin":   "CC(=O)OC1=CC=CC=C1C(=O)O",
        "Caffeine":  "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
        "Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
        "Glucose":   "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
        "Cholesterol":"[C@@H]1([C@H](CCCC(C)C)C)[C@H]2CC[C@@]3([H])[C@]4([H])CC=C5C[C@@H](O)CC[C@]5(C)[C@@H]4CC[C@]3(C)[C@@]2(CC1)C",
    }
    inference_results = []
    for name, smi in demo_molecules.items():
        print(f"\n  ── {name} ──")
        try:
            pred = predict_logsS(smi, model_path=model_save_path)
            inference_results.append(
                {"Molecule": name, "SMILES": smi, "Predicted LogS": round(pred, 4)}
            )
        except Exception as e:
            print(f"  ✗ Error: {e}")

    inf_df = pd.DataFrame(inference_results)
    inf_df.to_csv(OUTPUT_DIR / "11_inference_demo.csv", index=False)

    print("\n\n" + "═"*60)
    print(" PIPELINE COMPLETE — ALL OUTPUTS SAVED TO:", OUTPUT_DIR.resolve())
    print("═"*60)
    print(inf_df.to_string(index=False))

    return results_df, best_pipe, inf_df


if __name__ == "__main__":
    results_df, best_pipe, inf_df = main()
